# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Notebook 9B – Root-Aware Lexical Semantic Passage Retrieval (DD-011)

## Overview

This notebook implements the retrieval stage of the proposed
Root-Aware Lexical Semantic Retrieval Framework for Qur'an Question
Answering.

The authoritative query roots and Maqāyīs semantic evidence produced in
corrected Notebook 7 are consumed directly and matched against the
authoritative CAMeL-only passage representation from corrected Notebook 9A.

The retrieval process integrates morphological information,
validated Arabic roots, and semantic meanings derived from
*Maqāyīs al-Lughah* to identify candidate passages that are
lexically and semantically related to the user query.

The candidate passage lists generated in this notebook are ranked by the
six-feature LEM in Notebook 9C. No evaluation metric is computed here.


In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

from pathlib import Path
import hashlib
import json

import pandas as pd

from collections import defaultdict
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("=" * 70)
print("LIBRARIES LOADED")
print("=" * 70)
print("✓ pathlib")
print("✓ hashlib")
print("✓ json")
print("✓ pandas")
print("✓ collections.defaultdict")
print("✓ IPython.display")


LIBRARIES LOADED
✓ pathlib
✓ hashlib
✓ json
✓ pandas
✓ collections.defaultdict
✓ IPython.display


# Define Project Paths

This section defines the directories and file paths used throughout
the retrieval stage.

The notebook loads the linguistically enriched query representation
generated in corrected Notebook 7 and the validated passage representation
and vocabulary produced in corrected Notebook 9A. Portable workspace
discovery replaces the historical `03_Data`/`05_Experiments` paths.

Corrected candidates are exported to an isolated RALSR correction folder;
historical outputs are not overwritten.


In [2]:
# ============================================================
# Define Project Paths
# ============================================================

current_path = Path.cwd().resolve()
PROJECT_ROOT = None

for parent in [current_path] + list(current_path.parents):
    if (parent / "00_Project_Control").exists() and (parent / "02_Methodology_Evidence").exists():
        PROJECT_ROOT = parent
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the thesis workspace containing 00_Project_Control and 02_Methodology_Evidence."
    )

METHODOLOGY_DIR = PROJECT_ROOT / "02_Methodology_Evidence"

SEMANTIC_QUERY_FILE = (
    METHODOLOGY_DIR / "Maqāyīs Lookup" / "Corrected_Outputs" /
    "Notebook_7_Maqayis_Semantic_Enrichment" /
    "QuranQA_Semantic_Enrichment_Corrected.csv"
)

PASSAGE_CORRECTED_DIR = (
    METHODOLOGY_DIR / "QAC and Root Extraction" / "Corrected_Outputs" /
    "QPC_Passage_Side_Validation"
)

PASSAGE_REPRESENTATION_FILE = (
    PASSAGE_CORRECTED_DIR / "QuranQA_Enriched_Passage_Representation.csv"
)

PASSAGE_VOCABULARY_FILE = (
    PASSAGE_CORRECTED_DIR / "QuranQA_Linguistically_Enriched_Passages.csv"
)

OFFICIAL_QPC_FILE = (
    METHODOLOGY_DIR / "Data and Resources" / "QuranQA" /
    "QQA23_TaskA_QPC_v1.1.tsv"
)

CORRECTED_OUTPUT_DIR = (
    METHODOLOGY_DIR / "RALSR" / "Corrected_Outputs" / "RALSR_Lineage_Repair"
)
CORRECTED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_RESULTS_FILE = (
    CORRECTED_OUTPUT_DIR / "QuranQA_Candidate_Retrieval_Corrected.csv"
)

required_files = [
    SEMANTIC_QUERY_FILE,
    PASSAGE_REPRESENTATION_FILE,
    PASSAGE_VOCABULARY_FILE,
    OFFICIAL_QPC_FILE,
]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing authoritative input(s):\n" + "\n".join(map(str, missing_files)))

print("=" * 70)
print("AUTHORITATIVE RALSR PATHS")
print("=" * 70)
print(f"Project Root              : {PROJECT_ROOT}")
print(f"Notebook 7 Query Input    : {SEMANTIC_QUERY_FILE}")
print(f"Notebook 9A Passage Input : {PASSAGE_REPRESENTATION_FILE}")
print(f"Notebook 9A Vocabulary    : {PASSAGE_VOCABULARY_FILE}")
print(f"Official QPC              : {OFFICIAL_QPC_FILE}")
print(f"Corrected Candidate Output: {CANDIDATE_RESULTS_FILE}")


AUTHORITATIVE RALSR PATHS
Project Root              : ${THESIS_WORKSPACE}
Notebook 7 Query Input    : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Maqāyīs Lookup\Corrected_Outputs\Notebook_7_Maqayis_Semantic_Enrichment\QuranQA_Semantic_Enrichment_Corrected.csv
Notebook 9A Passage Input : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\QPC_Passage_Side_Validation\QuranQA_Enriched_Passage_Representation.csv
Notebook 9A Vocabulary    : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\QPC_Passage_Side_Validation\QuranQA_Linguistically_Enriched_Passages.csv
Official QPC              : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Data and Resources\QuranQA\QQA23_TaskA_QPC_v1.1.tsv
Corrected Candidate Output: ${THESIS_WORKSPACE}\02_Methodology_Evidence\RALSR\Corrected_Outputs\RALSR_Lineage_Repair\QuranQA_Candidate_Retrieval_Corrected.csv


# Build the Linguistic Representation of Qur'anic Passages

The passage representation is not rebuilt here. Corrected Notebook 9A has
already normalized and tokenized the official QPC passages, applied CAMeL-only
morphological analysis, validated roots, and built passage root lists from its
authoritative vocabulary.

Notebook 9B consumes that validated representation directly and constructs a
root-to-passage inverted index without reanalysing passage tokens.


# Load Retrieval Resources

This section loads the linguistically enriched query and passage
representations produced in previous notebooks.

Unlike the preprocessing notebooks, no additional linguistic analysis
is performed here. The retrieval engine operates directly on the
precomputed representations to efficiently identify and rank relevant
Qur'anic passages.


In [3]:
# ============================================================
# Load Authoritative Retrieval Resources
# ============================================================

semantic_queries_df = pd.read_csv(SEMANTIC_QUERY_FILE, encoding="utf-8-sig")
passage_representation_df = pd.read_csv(PASSAGE_REPRESENTATION_FILE, encoding="utf-8-sig")
passage_vocab_df = pd.read_csv(PASSAGE_VOCABULARY_FILE, encoding="utf-8-sig")
official_qpc_df = pd.read_csv(
    OFFICIAL_QPC_FILE,
    sep="\t",
    header=None,
    names=["Passage_ID", "Passage_Text"],
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

expected_query_columns = {
    "Dataset", "Question_ID", "Question", "Query_Terms",
    "Query_Term_Provenance", "Lexical_Matches"
}
expected_passage_columns = {
    "Passage_ID", "Passage_Text", "Roots", "Semantic_Meanings",
    "Root_Count", "Semantic_Count"
}

if not expected_query_columns.issubset(semantic_queries_df.columns):
    raise ValueError(f"Notebook 7 input is missing columns: {sorted(expected_query_columns - set(semantic_queries_df.columns))}")
if not expected_passage_columns.issubset(passage_representation_df.columns):
    raise ValueError(f"Notebook 9A input is missing columns: {sorted(expected_passage_columns - set(passage_representation_df.columns))}")

official_qpc_ids = set(official_qpc_df["Passage_ID"].astype(str))
passage_ids = passage_representation_df["Passage_ID"].astype(str)

assert len(official_qpc_df) == 1266
assert official_qpc_df["Passage_ID"].nunique() == 1266
assert len(passage_representation_df) == 1266
assert passage_ids.nunique() == 1266
assert set(passage_ids) == official_qpc_ids

print("=" * 70)
print("AUTHORITATIVE RETRIEVAL RESOURCES LOADED")
print("=" * 70)
print(f"Semantic Query Records : {len(semantic_queries_df):,}")
print(f"Official QPC Passages   : {len(official_qpc_df):,}")
print(f"Passage Records         : {len(passage_representation_df):,}")
print(f"Vocabulary Terms        : {len(passage_vocab_df):,}")
print("✓ Corrected Notebook 9A passage IDs exactly match the official QPC")


AUTHORITATIVE RETRIEVAL RESOURCES LOADED
Semantic Query Records : 251
Official QPC Passages   : 1,266
Passage Records         : 1,266
Vocabulary Terms        : 14,693
✓ Corrected Notebook 9A passage IDs exactly match the official QPC


# Build the Root Inverted Index

To support efficient passage retrieval, an inverted index is
constructed using the validated Arabic roots extracted from each
Qur'anic passage.

The index maps every unique Arabic root to the set of passages in
which it occurs. During retrieval, candidate passages can therefore
be located directly from the query roots without scanning the entire
passage collection.

This root-based index forms the primary retrieval structure of the
Root-Aware Lexical Semantic Retrieval framework.

In [4]:
# ============================================================
# Build Root Inverted Index from Corrected Notebook 9A
# ============================================================

def parse_json_list(value, field_name):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, list):
        return value
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError(f"{field_name} must contain a JSON list")
    return parsed


root_index = defaultdict(set)

for _, row in passage_representation_df.iterrows():
    passage_id = str(row["Passage_ID"])
    roots = parse_json_list(row["Roots"], "Roots")

    for root in roots:
        root = "" if root is None else str(root).strip()
        if root:
            root_index[root].add(passage_id)

indexed_passage_ids = set().union(*root_index.values()) if root_index else set()
assert indexed_passage_ids.issubset(official_qpc_ids)

print("=" * 70)
print("AUTHORITATIVE ROOT INVERTED INDEX CREATED")
print("=" * 70)
print(f"Unique Roots Indexed : {len(root_index):,}")
print(f"Root → Passage Links : {sum(len(v) for v in root_index.values()):,}")
print(f"Passages Indexed     : {len(indexed_passage_ids):,}")
print("✓ No passage root was reconstructed")


AUTHORITATIVE ROOT INVERTED INDEX CREATED
Unique Roots Indexed : 1,202
Root → Passage Links : 30,683
Passages Indexed     : 1,266
✓ No passage root was reconstructed


# Validate Passage-Root Authority

The corrected Notebook 9A vocabulary is retained only as passage-lineage
evidence. It is not a canonical lookup for query terms. Every passage root in
the corrected passage representation must occur among the vocabulary records
that Notebook 9A marked as valid and resolved.


In [5]:
# ============================================================
# Validate Corrected Notebook 9A Passage-Root Lineage
# ============================================================

resolved_vocab = passage_vocab_df.loc[
    passage_vocab_df["Root_Resolved"].fillna(False).astype(bool)
].copy()

valid_vocabulary_roots = {
    str(root).strip()
    for root in resolved_vocab["Root_AR"].dropna()
    if str(root).strip()
}

representation_roots = set(root_index.keys())
passage_root_lineage_mismatches = sorted(representation_roots - valid_vocabulary_roots)

assert not passage_root_lineage_mismatches, (
    "Passage representation contains roots absent from the validated Notebook 9A vocabulary: "
    f"{passage_root_lineage_mismatches[:10]}"
)

print("=" * 70)
print("PASSAGE ROOT LINEAGE VALIDATED")
print("=" * 70)
print(f"Validated Vocabulary Roots : {len(valid_vocabulary_roots):,}")
print(f"Representation Roots       : {len(representation_roots):,}")
print(f"Lineage Mismatches          : {len(passage_root_lineage_mismatches):,}")
print("✓ Passage vocabulary is validation evidence only, never query-root authority")


PASSAGE ROOT LINEAGE VALIDATED
Validated Vocabulary Roots : 1,202
Representation Roots       : 1,202
Lineage Mismatches          : 0
✓ Passage vocabulary is validation evidence only, never query-root authority


Candidate Passage Retrieval
The Root Inverted Index is used to efficiently retrieve an initial set of candidate passages for each user query.

Rather than scanning the complete Qur'anic passage collection, the retrieval process identifies only those passages that contain one or more of the validated Arabic roots extracted from the query.

This candidate generation stage substantially reduces the search space while preserving passages that are lexically related to the user's information need.


# Retrieve Candidate Passages

Candidate passages are retrieved from the Root Inverted Index using the
authoritative query roots stored in corrected Notebook 7. The historical
passage-vocabulary query lookup, prefix stripping, and downstream query-root
reconstruction are absent from the corrected path.

Unresolved query terms remain unresolved. The candidate set remains the union
of official QPC passages containing one or more authoritative query roots and
forms the input to Notebook 9C.


In [6]:
# ============================================================
# Authoritative Query-Lineage Utilities
# ============================================================

def ordered_unique(values):
    return list(dict.fromkeys(values))


def compact_root_record(item):
    return {
        "Query_Term": item.get("Query_Term"),
        "Original_Term": item.get("Original_Term"),
        "Normalized_Term": item.get("Normalized_Term"),
        "Root_AR": item.get("Root_AR"),
        "Root_Source": item.get("Root_Source"),
        "Resolution_Status": item.get("Resolution_Status"),
        "Semantic_Found": bool(item.get("Semantic_Found", False)),
        "Semantic_Source": item.get("Semantic_Source"),
    }


print("=" * 70)
print("AUTHORITATIVE QUERY-LINEAGE UTILITIES READY")
print("=" * 70)
print("✓ No passage-vocabulary query lookup exists in the corrected path")


AUTHORITATIVE QUERY-LINEAGE UTILITIES READY
✓ No passage-vocabulary query lookup exists in the corrected path


In [7]:
# ============================================================
# Query-Root Authority Guard
# ============================================================

def validate_authoritative_match(item):
    resolved = bool(item.get("Root_Resolved", False))
    root = item.get("Root_AR")
    source = item.get("Root_Source")

    root_text = "" if root is None else str(root).strip()
    source_text = "" if source is None else str(source).strip()

    if resolved and (not root_text or not source_text):
        raise ValueError(
            "Notebook 7 contains a resolved query record without a root or Root_Source"
        )
    if not resolved and root_text:
        raise ValueError(
            "Notebook 7 contains an unresolved query record with a selected root"
        )

    return resolved, root_text, source_text


print("=" * 70)
print("QUERY-ROOT AUTHORITY GUARD READY")
print("=" * 70)


QUERY-ROOT AUTHORITY GUARD READY


In [8]:
# ============================================================
# Retrieve Candidate Passages
# ============================================================

candidate_results = []

for _, query in semantic_queries_df.iterrows():
    question_id = query["Question_ID"]
    question = query["Question"]
    dataset = query["Dataset"]

    query_terms = parse_json_list(query["Query_Terms"], "Query_Terms")
    query_term_provenance = parse_json_list(
        query["Query_Term_Provenance"], "Query_Term_Provenance"
    )
    lexical_matches = parse_json_list(query["Lexical_Matches"], "Lexical_Matches")

    query_roots = []
    query_semantics = []
    query_root_records = []

    for item in lexical_matches:
        resolved, root, source = validate_authoritative_match(item)
        if not resolved:
            continue

        query_roots.append(root)
        query_root_records.append(compact_root_record(item))

        semantic = item.get("Semantic_Meaning")
        if bool(item.get("Semantic_Found", False)) and semantic is not None:
            semantic = str(semantic).strip()
            if semantic:
                query_semantics.append(semantic)

    query_roots = ordered_unique(query_roots)
    query_semantics = ordered_unique(query_semantics)

    candidate_passages = set()
    for root in query_roots:
        candidate_passages.update(root_index.get(root, set()))

    invalid_candidate_ids = candidate_passages - official_qpc_ids
    if invalid_candidate_ids:
        raise ValueError(f"Candidate IDs outside official QPC: {sorted(invalid_candidate_ids)[:10]}")

    candidate_results.append({
        "Dataset": dataset,
        "Question_ID": question_id,
        "Question": question,
        "Query_Terms": query_terms,
        "Query_Term_Provenance": query_term_provenance,
        "Query_Lexical_Matches": lexical_matches,
        "Query_Root_Records": query_root_records,
        "Query_Roots": query_roots,
        "Query_Semantics": query_semantics,
        "Candidate_Passages": sorted(candidate_passages),
        "Candidate_Count": len(candidate_passages),
    })

candidate_df = pd.DataFrame(candidate_results)

assert len(candidate_df) == len(semantic_queries_df)
assert candidate_df["Question_ID"].nunique() == semantic_queries_df["Question_ID"].nunique()

print("=" * 70)
print("AUTHORITATIVE CANDIDATE RETRIEVAL COMPLETED")
print("=" * 70)
print(f"Questions Processed : {len(candidate_df):,}")
print(f"Average Candidates  : {candidate_df['Candidate_Count'].mean():.2f}")
print(f"Zero-Candidate QIDs : {(candidate_df['Candidate_Count'] == 0).sum():,}")
display(candidate_df[["Question_ID", "Query_Roots", "Candidate_Count"]].head(10))


AUTHORITATIVE CANDIDATE RETRIEVAL COMPLETED
Questions Processed : 251
Average Candidates  : 233.54
Zero-Candidate QIDs : 13


,Question_ID,Query_Roots,Candidate_Count
0,101,"[قوم, شعب]",7
1,102,[قوم],0
2,103,"[بني, كعب]",4
3,105,"[نبا, عرف, صبر]",125
4,106,"[كفل, سدة]",7
5,107,[حطم],5
6,108,[اخو],0
7,109,[قرع],3
8,110,[جثو],0
9,111,[سبط],5


In [9]:
# ============================================================
# Validate and Export Corrected Candidate Retrieval Results
# ============================================================

json_columns = [
    "Query_Terms",
    "Query_Term_Provenance",
    "Query_Lexical_Matches",
    "Query_Root_Records",
    "Query_Roots",
    "Query_Semantics",
    "Candidate_Passages",
]

export_candidate_df = candidate_df.copy()
for column in json_columns:
    export_candidate_df[column] = export_candidate_df[column].map(
        lambda value: json.dumps(value, ensure_ascii=False, allow_nan=False)
    )

export_candidate_df.to_csv(
    CANDIDATE_RESULTS_FILE,
    index=False,
    encoding="utf-8-sig",
)

reloaded_candidate_df = pd.read_csv(CANDIDATE_RESULTS_FILE, encoding="utf-8-sig")
assert len(reloaded_candidate_df) == len(candidate_df)
for column in json_columns:
    reloaded_candidate_df[column].map(json.loads)

candidate_sha256 = hashlib.sha256(CANDIDATE_RESULTS_FILE.read_bytes()).hexdigest().upper()

print("=" * 70)
print("CORRECTED CANDIDATE RETRIEVAL EXPORTED")
print("=" * 70)
print(f"Questions : {len(candidate_df):,}")
print(f"Output    : {CANDIDATE_RESULTS_FILE}")
print(f"SHA-256   : {candidate_sha256}")
print("✓ Strict JSON fields reload successfully")


CORRECTED CANDIDATE RETRIEVAL EXPORTED
Questions : 251
Output    : ${THESIS_WORKSPACE}\02_Methodology_Evidence\RALSR\Corrected_Outputs\RALSR_Lineage_Repair\QuranQA_Candidate_Retrieval_Corrected.csv
SHA-256   : 0945FBB7A1B2CE76908667DFF478CCA037CCBFB291D69986D3DDE3DE06E007A7
✓ Strict JSON fields reload successfully
